# CNC RAG 消融实验：Phase 2（Qwen3.6 27B No Thinking）

本 notebook 使用 `local/qwen3.6-27b-no-thinking`，在固定 `cnc_sft_validation`（500 条）上复现与 Gemma 31B 相同的 Phase 2 消融配置，以独立确定 Qwen 的 k 值。

| 配置 | k | 动态示例 | 总示例 |
|---|---:|---:|---:|
| fixed | 0 | 0 | 10 |
| RAG only | 1 / 3 / 5 | 2k | 2 / 6 / 10 |
| fixed + random | 1 / 3 / 5 | 2k | 12 / 16 / 20 |
| fixed + RAG | 1 / 3 / 5 | 2k | 12 / 16 / 20 |

`2k` 来自 Pattern top-k + KNN top-k。若两路结果重叠，检索器会从后续候选补足，确保与 random 的数量严格一致。

RAG 与 random 使用同一份 600 条 train-only CNC support；它与 validation/test 均无 ID 重叠。该 support 全部为正因果样本，而 fixed 10 examples 包含 5 正、5 负，因此 RAG-only 与 fixed 的数量在 k=5 时匹配，但类别构成不同。

模型只加载一次，10 个配置依次运行，最后自动卸载。每完成一个配置都会保存独立 Markdown 报告并增量更新汇总 CSV。


In [1]:
# ===== 全局配置 =====
import json
import logging
import sys
from collections.abc import Iterable, Iterator
from html import escape
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if 'master_thesis' not in sys.executable.lower():
    raise RuntimeError('请切换到 Master_thesis kernel，重启 kernel 后从第一格重新运行。')

from src.data_io import load_dataset
from src.eval_pipeline import EvalRunConfig, run_stream_eval
from src.llm_client import LLMClient
from src.prompt_builder import build_messages, load_prompt_template
from src.retriever import (
    DeterministicRandomRetriever,
    ExactCountHybridRetriever,
    KNNRetriever,
    PatternRetriever,
    load_examples_from_jsonl,
    resolve_rag_cache_paths,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
LOGGER = logging.getLogger('lmstudio_rag_ablation_phase2_qwen27')
logging.getLogger('src.llm_client').setLevel(logging.WARNING)


def _notebook_progress(
    iterable: Iterable[Any],
    *,
    total: int,
    desc: str,
) -> Iterator[Any]:
    safe_desc = escape(desc)
    safe_total = max(total, 1)

    def render(completed: int) -> HTML:
        percent = min(completed / safe_total * 100, 100.0)
        return HTML(
            f"<div style='width:100%'>"
            f"<div>{safe_desc}: {completed}/{total} ({percent:.1f}%)</div>"
            f"<progress value='{completed}' max='{safe_total}' "
            f"style='width:100%;height:18px'></progress></div>"
        )

    handle = display(render(0), display_id=True)
    for completed, item in enumerate(iterable, 1):
        yield item
        if handle is not None:
            handle.update(render(completed))


LMSTUDIO_BASE_URL = 'http://127.0.0.1:1234/v1'
LMSTUDIO_API_KEY = 'lm-studio'
MODEL_KEY = 'local/qwen3.6-27b-no-thinking'
MODEL_DISPLAY_NAME = 'Qwen3.6 27B No Thinking'
MODEL_LOAD_TIMEOUT = 1200
LLM_TIMEOUT = 600
LLM_RETRY_TIMES = 3
CONTEXT_LENGTH = 8192
MODEL_PARALLEL = 1
OFFLOAD_KV_CACHE_TO_GPU = True
MAX_TOKENS = 2048
TEMPERATURE = 0.0

DATASET_NAME = 'cnc_sft_validation'
EVAL_SAMPLE_N = None  # None = 完整 validation 500 条。
EVAL_PROGRESS_EVERY = 500
EVAL_MAX_WORKERS = 1
K_VALUES = (1, 3, 5)
RANDOM_SEED = 42
FIXED_PROMPT = 'v9.6'
ZERO_SHOT_PROMPT = 'v9.6_zero_shot'
REPORT_DIR = Path('results') / 'eval_report' / 'rag_ablation' / 'phase2_qwen27'
RUN_PHASE2 = False  # 原始十组已完成；补跑只使用 notebook 末尾的专用开关。

EXPERIMENT_RUNS: list[dict[str, Any]] = [
    {
        'run_id': 'fixed',
        'configuration': 'fixed',
        'k': 0,
        'prompt_name': FIXED_PROMPT,
        'retriever_kind': 'none',
        'rag_mode': 'off',
        'dynamic_examples': 0,
        'total_examples': 10,
    },
]
for k in K_VALUES:
    EXPERIMENT_RUNS.append({
        'run_id': f'rag_only_k{k}',
        'configuration': 'RAG only',
        'k': k,
        'prompt_name': ZERO_SHOT_PROMPT,
        'retriever_kind': 'rag',
        'rag_mode': 'knn_pattern',
        'dynamic_examples': 2 * k,
        'total_examples': 2 * k,
    })
for k in K_VALUES:
    EXPERIMENT_RUNS.append({
        'run_id': f'fixed_random_k{k}',
        'configuration': 'fixed + random',
        'k': k,
        'prompt_name': FIXED_PROMPT,
        'retriever_kind': 'random',
        'rag_mode': 'random',
        'dynamic_examples': 2 * k,
        'total_examples': 10 + 2 * k,
    })
for k in K_VALUES:
    EXPERIMENT_RUNS.append({
        'run_id': f'fixed_rag_k{k}',
        'configuration': 'fixed + RAG',
        'k': k,
        'prompt_name': FIXED_PROMPT,
        'retriever_kind': 'rag',
        'rag_mode': 'knn_pattern',
        'dynamic_examples': 2 * k,
        'total_examples': 10 + 2 * k,
    })

display(pd.DataFrame(EXPERIMENT_RUNS))


,run_id,configuration,k,prompt_name,retriever_kind,rag_mode,dynamic_examples,total_examples
0,fixed,fixed,0,v9.6,none,off,0,10
1,rag_only_k1,RAG only,1,v9.6_zero_shot,rag,knn_pattern,2,2
2,rag_only_k3,RAG only,3,v9.6_zero_shot,rag,knn_pattern,6,6
3,rag_only_k5,RAG only,5,v9.6_zero_shot,rag,knn_pattern,10,10
4,fixed_random_k1,fixed + random,1,v9.6,random,random,2,12
5,fixed_random_k3,fixed + random,3,v9.6,random,random,6,16
6,fixed_random_k5,fixed + random,5,v9.6,random,random,10,20
7,fixed_rag_k1,fixed + RAG,1,v9.6,rag,knn_pattern,2,12
8,fixed_rag_k3,fixed + RAG,3,v9.6,rag,knn_pattern,6,16
9,fixed_rag_k5,fixed + RAG,5,v9.6,rag,knn_pattern,10,20


In [2]:
# ===== 模型、数据隔离、prompt 与 2k 数量预检（不加载 LLM） =====
manager = LLMClient(
    provider='lmstudio',
    base_url=LMSTUDIO_BASE_URL,
    model=MODEL_KEY,
    api_key=LMSTUDIO_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    context_length=CONTEXT_LENGTH,
    reasoning='off',
    timeout=MODEL_LOAD_TIMEOUT,
    retry_times=LLM_RETRY_TIMES,
)

inventory_by_key = {
    str(item['key']): item
    for item in manager.list_local_models()
    if item.get('key')
}
if MODEL_KEY not in inventory_by_key:
    raise RuntimeError(f'LM Studio 中找不到目标模型：{MODEL_KEY}')
model_item = inventory_by_key[MODEL_KEY]
reasoning_default = model_item.get('capabilities', {}).get('reasoning', {}).get('default')
if reasoning_default != 'off':
    raise RuntimeError(f'{MODEL_KEY} 的 Enable Thinking 默认值不是 off：{reasoning_default}')
display(pd.DataFrame([{
    'model': MODEL_DISPLAY_NAME,
    'model_key': MODEL_KEY,
    'quantization': model_item.get('quantization', {}).get('name'),
    'params': model_item.get('params_string'),
    'reasoning_default': reasoning_default,
    'loaded_instances': len(model_item.get('loaded_instances', [])),
}]))

validation_samples = load_dataset(DATASET_NAME, n=EVAL_SAMPLE_N)
validation_ids = {str(sample['id']) for sample in validation_samples}
metadata_path, embeddings_path = resolve_rag_cache_paths('cnc')
support_examples = load_examples_from_jsonl(metadata_path)
support_ids = {str(example['sample_id']) for example in support_examples}
overlap = validation_ids.intersection(support_ids)
if overlap:
    raise RuntimeError(f'CNC RAG support 与 validation 有 {len(overlap)} 个精确 ID 重叠。')
if any(not example.get('triples') for example in support_examples):
    raise RuntimeError('CNC RAG support 中存在没有 causal triples 的样本。')

manifest = json.loads(
    (PROJECT_ROOT / 'RAG Database/cnc_split_manifest.json').read_text(encoding='utf-8')
)
if not manifest.get('train_only_support'):
    raise RuntimeError('CNC RAG manifest 未声明 train_only_support。')
if int(manifest.get('summary', {}).get('validation_sample_overlap', -1)) != 0:
    raise RuntimeError('CNC RAG manifest 显示 validation overlap 非零。')

fixed_template = load_prompt_template(FIXED_PROMPT)
zero_template = load_prompt_template(ZERO_SHOT_PROMPT)
if fixed_template.count('\nInput:\n') != 10:
    raise RuntimeError('v9.6 fixed prompt 不包含恰好 10 个 fixed examples。')
if zero_template.count('\nInput:\n') != 0:
    raise RuntimeError('v9.6 zero-shot prompt 仍包含 fixed examples。')
if fixed_template.count('{rag_examples}') != 1:
    raise RuntimeError('v9.6 必须包含且只包含一个 {rag_examples} 占位符。')
if zero_template.count('{rag_examples}') != 1:
    raise RuntimeError('v9.6 zero-shot prompt 必须包含且只包含一个 {rag_examples} 占位符。')

RAG_RETRIEVER = ExactCountHybridRetriever(
    pattern_retriever=PatternRetriever(metadata_path=metadata_path),
    knn_retriever=KNNRetriever(
        metadata_path=metadata_path,
        embeddings_path=embeddings_path,
        device='cpu',
    ),
)
RANDOM_RETRIEVER = DeterministicRandomRetriever(
    metadata_path=metadata_path,
    seed=RANDOM_SEED,
)

audit_text = 'Heavy rain caused flooding in the region.'
count_rows: list[dict[str, Any]] = []
for k in K_VALUES:
    rag_examples = RAG_RETRIEVER.retrieve(audit_text, top_k=k)
    random_examples = RANDOM_RETRIEVER.retrieve(audit_text, top_k=k)
    count_rows.append({
        'k': k,
        'expected_each': 2 * k,
        'rag_examples': len(rag_examples),
        'random_examples': len(random_examples),
    })
    for retriever_kind, retriever, rag_mode in (
        ('rag', RAG_RETRIEVER, 'knn_pattern'),
        ('random', RANDOM_RETRIEVER, 'random'),
    ):
        messages = build_messages(
            audit_text,
            use_rag=True,
            retriever=retriever,
            top_k=k,
            rag_mode=rag_mode,
            prompt_name=FIXED_PROMPT,
        )
        system_prompt = messages[0]['content']
        inserted_examples = system_prompt.count('\nExample ')
        if inserted_examples != 2 * k or '{rag_examples}' in system_prompt:
            raise RuntimeError(
                f'v9.6 动态示例注入失败：{retriever_kind=} {k=} '
                f'expected={2 * k} actual={inserted_examples}'
            )
display(pd.DataFrame(count_rows))
LOGGER.info(
    '预检通过：validation=%s support=%s overlap=0 fixed=10 zero-shot=0；RAG/random 均严格返回 2k。',
    len(validation_samples),
    len(support_examples),
)


,model,model_key,quantization,params,reasoning_default,loaded_instances
0,Qwen3.6 27B No Thinking,local/qwen3.6-27b-no-thinking,Q4_K_M,27B,off,0


2026-08-27 00:40:13,808 | INFO | Pattern examples 已加载：path=D:\Master thesis\RAG Database\cnc_examples.jsonl examples=600
2026-08-27 00:40:13,820 | INFO | KNN cache 已加载：examples=600 path=D:\Master thesis\RAG Database\cnc_examples.jsonl embedding_device=cpu
D:\Anaconda3\envs\Master_thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-27 00:40:24,890 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-27 00:40:24,938 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-08-27 00:40:25,092 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transform

,k,expected_each,rag_examples,random_examples
0,1,2,2,2
1,3,6,6,6
2,5,10,10,10


2026-08-27 00:40:28,969 | INFO | 预检通过：validation=500 support=600 overlap=0 fixed=10 zero-shot=0；RAG/random 均严格返回 2k。


In [3]:
# ===== Phase 2 评测辅助函数 =====
SUMMARY_PATH = PROJECT_ROOT / REPORT_DIR / 'phase2_qwen27_summary.csv'


def _retriever_for_run(run: dict[str, Any]) -> Any:
    if run['retriever_kind'] == 'none':
        return None
    if run['retriever_kind'] == 'random':
        return RANDOM_RETRIEVER
    if run['retriever_kind'] == 'rag':
        return RAG_RETRIEVER
    raise ValueError(f"未知 retriever_kind：{run['retriever_kind']}")


def _summary_row(
    run: dict[str, Any],
    report: dict[str, Any],
    *,
    context_length: int = CONTEXT_LENGTH,
    parallel: int = MODEL_PARALLEL,
) -> dict[str, Any]:
    detection = report['detection']
    strict = report['extraction']['strict_token_f1']
    anchor = report['extraction']['anchor_window']
    return {
        'status': 'completed',
        'run_id': run['run_id'],
        'configuration': run['configuration'],
        'k': run['k'],
        'dynamic_examples': run['dynamic_examples'],
        'total_examples': run['total_examples'],
        'split': 'validation',
        'model': MODEL_DISPLAY_NAME,
        'model_key': MODEL_KEY,
        'prompt': run['prompt_name'],
        'rag_mode': run['rag_mode'],
        'random_seed': RANDOM_SEED if run['retriever_kind'] == 'random' else '',
        'temperature': TEMPERATURE,
        'context_length': context_length,
        'max_tokens': MAX_TOKENS,
        'parallel': parallel,
        'offload_kv_cache_to_gpu': OFFLOAD_KV_CACHE_TO_GPU,
        'reasoning': 'model_default_off',
        'detection_accuracy': detection['accuracy'],
        'detection_precision': detection['precision'],
        'detection_recall': detection['recall'],
        'detection_f1': detection['f1'],
        'strict_all_precision': strict['all_samples']['precision'],
        'strict_all_recall': strict['all_samples']['recall'],
        'strict_all_f1': strict['all_samples']['f1'],
        'anchor_all_precision': anchor['all_samples']['precision'],
        'anchor_all_recall': anchor['all_samples']['recall'],
        'anchor_all_f1': anchor['all_samples']['f1'],
        'strict_detected_only_f1': strict['detected_only']['f1'],
        'anchor_detected_only_f1': anchor['detected_only']['f1'],
        'report_path': report.get('report_path', ''),
        'error': '',
    }


def _save_summary(
    rows: list[dict[str, Any]],
    summary_path: Path = SUMMARY_PATH,
) -> None:
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(summary_path, index=False, encoding='utf-8-sig')
    LOGGER.info('Phase 2 增量汇总已保存：%s', summary_path)


def run_phase2(
    experiment_runs: list[dict[str, Any]],
    *,
    summary_path: Path = SUMMARY_PATH,
    context_length: int = CONTEXT_LENGTH,
    parallel: int = MODEL_PARALLEL,
) -> list[dict[str, Any]]:
    samples = load_dataset(DATASET_NAME, n=EVAL_SAMPLE_N)
    rows: list[dict[str, Any]] = []
    manager.unload_all_models()
    load_result = manager.load_model(
        MODEL_KEY,
        context_length=context_length,
        parallel=parallel,
        offload_kv_cache_to_gpu=OFFLOAD_KV_CACHE_TO_GPU,
    )
    load_config = load_result.get('load_config', {})
    actual_context = int(load_config.get('context_length', 0))
    actual_parallel = int(load_config.get('parallel', 0))
    actual_kv_cache_gpu = load_config.get('offload_kv_cache_to_gpu') is True
    if (
        actual_context != context_length
        or actual_parallel != parallel
        or actual_kv_cache_gpu != OFFLOAD_KV_CACHE_TO_GPU
    ):
        manager.unload_all_models()
        raise RuntimeError(
            '模型实际加载配置与目标不一致：'
            f'context={actual_context}/{context_length}, '
            f'parallel={actual_parallel}/{parallel}, '
            f'kv_cache_gpu={actual_kv_cache_gpu}/{OFFLOAD_KV_CACHE_TO_GPU}。'
        )
    instance_id = str(load_result['instance_id'])
    loaded_instance = next(
        (
            instance
            for model_item in manager.list_local_models()
            if model_item.get('key') == MODEL_KEY
            for instance in model_item.get('loaded_instances', [])
            if isinstance(instance, dict) and instance.get('id') == instance_id
        ),
        None,
    )
    inventory_config = (
        loaded_instance.get('config', {})
        if isinstance(loaded_instance, dict)
        else {}
    )
    inventory_context = int(inventory_config.get('context_length', 0))
    inventory_parallel = int(inventory_config.get('parallel', 0))
    if inventory_context != context_length or inventory_parallel != parallel:
        manager.unload_all_models()
        raise RuntimeError(
            'LM Studio inventory 中的实例配置与目标不一致：'
            f'context={inventory_context}/{context_length}, '
            f'parallel={inventory_parallel}/{parallel}。'
        )
    LOGGER.info(
        'LM Studio 实例已确认：id=%s context=%s parallel=%s',
        instance_id,
        inventory_context,
        inventory_parallel,
    )
    client = LLMClient(
        provider='lmstudio',
        base_url=LMSTUDIO_BASE_URL,
        model=instance_id,
        api_key=LMSTUDIO_API_KEY,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        context_length=context_length,
        timeout=LLM_TIMEOUT,
        retry_times=LLM_RETRY_TIMES,
    )

    try:
        for position, run in enumerate(experiment_runs, 1):
            LOGGER.info(
                '[Phase 2 %s/%s] 开始 %s',
                position,
                len(experiment_runs),
                run['run_id'],
            )
            try:
                use_rag = run['retriever_kind'] != 'none'
                retriever = _retriever_for_run(run)
                eval_config = EvalRunConfig(
                    project_root=PROJECT_ROOT,
                    model=instance_id,
                    dataset=DATASET_NAME,
                    prompt_name=str(run['prompt_name']),
                    use_rag=use_rag,
                    rag_mode=str(run['rag_mode']),
                    rag_top_k=int(run['k']),
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    primary_metric='strict_token_f1',
                    progress_every=EVAL_PROGRESS_EVERY,
                    max_workers=EVAL_MAX_WORKERS,
                    llm_provider='lmstudio',
                    llm_base_url=LMSTUDIO_BASE_URL,
                    context_length=context_length,
                    reasoning_effort=None,
                    llm_extra_body=None,
                    api_key_source='lmstudio-default',
                    save_report=True,
                    report_dir=REPORT_DIR,
                    report_detail_limit=200,
                    report_detail_mode='errors',
                    report_error_metric='strict_token_f1',
                    metadata_path=metadata_path if use_rag else None,
                    embeddings_path=(
                        embeddings_path if run['retriever_kind'] == 'rag' else None
                    ),
                )
                report = run_stream_eval(
                    samples=samples,
                    label=f"{MODEL_DISPLAY_NAME} CNC validation {run['run_id']}",
                    client=client,
                    config=eval_config,
                    existing_retriever=retriever,
                    progress_factory=_notebook_progress,
                    emit=LOGGER.info,
                )
                rows.append(
                    _summary_row(
                        run,
                        report,
                        context_length=context_length,
                        parallel=parallel,
                    )
                )
            except Exception as exc:
                LOGGER.exception('[Phase 2 %s/%s] %s 失败', position, len(experiment_runs), run['run_id'])
                rows.append({
                    'status': 'failed',
                    'run_id': run['run_id'],
                    'configuration': run['configuration'],
                    'k': run['k'],
                    'dynamic_examples': run['dynamic_examples'],
                    'total_examples': run['total_examples'],
                    'split': 'validation',
                    'model': MODEL_DISPLAY_NAME,
                    'model_key': MODEL_KEY,
                    'prompt': run['prompt_name'],
                    'rag_mode': run['rag_mode'],
                    'error': f'{type(exc).__name__}: {exc}',
                })
            _save_summary(rows, summary_path)
    finally:
        manager.unload_all_models()
    return rows


In [ ]:
# ===== 运行 Phase 2：Qwen3.6 27B 的 10 个消融配置 =====
if not RUN_PHASE2:
    LOGGER.warning('Phase 2 未启动。确认预检结果后，将 RUN_PHASE2 改为 True。')
else:
    PHASE2_RESULTS = run_phase2(EXPERIMENT_RUNS)
    phase2_summary_df = pd.DataFrame(PHASE2_RESULTS)
    display(phase2_summary_df)


## 修正后补跑：仅重跑六组 fixed + dynamic 配置

原始 `v9.6` 缺少 `{rag_examples}` 占位符，导致先前的 `fixed + random` 与 `fixed + RAG` 没有真正注入动态示例。修正 prompt 后，本节仅补跑受影响的六组；`fixed` 与三组 `RAG only` 结果仍然有效，不重复运行。补跑汇总单独保存，不覆盖原始十组汇总。


In [4]:
# ===== 修正后补跑配置：只包含六组受占位符问题影响的实验 =====
RERUN_RUN_IDS = (
    'fixed_random_k1',
    'fixed_random_k3',
    'fixed_random_k5',
    'fixed_rag_k1',
    'fixed_rag_k3',
    'fixed_rag_k5',
)
RERUN_SUMMARY_PATH = (
    PROJECT_ROOT / REPORT_DIR / 'phase2_qwen27_rerun_summary.csv'
)
CORRECTED_SUMMARY_PATH = (
    PROJECT_ROOT / REPORT_DIR / 'phase2_qwen27_corrected_summary.csv'
)
VALID_ORIGINAL_RUN_IDS = ('fixed', 'rag_only_k1', 'rag_only_k3', 'rag_only_k5')
RUN_PHASE2_RERUN = False  # 前五组已完成；不要再次运行原六组入口。

runs_by_id = {run['run_id']: run for run in EXPERIMENT_RUNS}
RERUN_EXPERIMENT_RUNS = [runs_by_id[run_id] for run_id in RERUN_RUN_IDS]
if len(RERUN_EXPERIMENT_RUNS) != 6:
    raise RuntimeError('补跑配置必须恰好包含六组。')
if any(run['retriever_kind'] == 'none' for run in RERUN_EXPERIMENT_RUNS):
    raise RuntimeError('补跑中不应包含 fixed 或 RAG-only 基线。')
if load_prompt_template(FIXED_PROMPT).count('{rag_examples}') != 1:
    raise RuntimeError('v9.6 的 {rag_examples} 占位符修正尚未生效。')

display(pd.DataFrame(RERUN_EXPERIMENT_RUNS))
print(f'补跑汇总将保存到：{RERUN_SUMMARY_PATH}')
print(f'修正版完整汇总将保存到：{CORRECTED_SUMMARY_PATH}')


,run_id,configuration,k,prompt_name,retriever_kind,rag_mode,dynamic_examples,total_examples
0,fixed_random_k1,fixed + random,1,v9.6,random,random,2,12
1,fixed_random_k3,fixed + random,3,v9.6,random,random,6,16
2,fixed_random_k5,fixed + random,5,v9.6,random,random,10,20
3,fixed_rag_k1,fixed + RAG,1,v9.6,rag,knn_pattern,2,12
4,fixed_rag_k3,fixed + RAG,3,v9.6,rag,knn_pattern,6,16
5,fixed_rag_k5,fixed + RAG,5,v9.6,rag,knn_pattern,10,20


补跑汇总将保存到：D:\Master thesis\results\eval_report\rag_ablation\phase2_qwen27\phase2_qwen27_rerun_summary.csv
修正版完整汇总将保存到：D:\Master thesis\results\eval_report\rag_ablation\phase2_qwen27\phase2_qwen27_corrected_summary.csv


In [5]:
# ===== 执行修正后的六组补跑 =====
if not RUN_PHASE2_RERUN:
    LOGGER.warning('补跑未启动。确认配置后，将 RUN_PHASE2_RERUN 改为 True。')
else:
    PHASE2_RERUN_RESULTS = run_phase2(
        RERUN_EXPERIMENT_RUNS,
        summary_path=RERUN_SUMMARY_PATH,
    )
    phase2_rerun_summary_df = pd.DataFrame(PHASE2_RERUN_RESULTS)
    display(phase2_rerun_summary_df)

    original_summary_df = pd.read_csv(SUMMARY_PATH)
    valid_original_df = original_summary_df[
        original_summary_df['run_id'].isin(VALID_ORIGINAL_RUN_IDS)
    ].copy()
    if set(valid_original_df['run_id']) != set(VALID_ORIGINAL_RUN_IDS):
        raise RuntimeError('原始汇总中缺少 fixed 或 RAG-only 有效结果，无法合并。')
    corrected_summary_df = pd.concat(
        [valid_original_df, phase2_rerun_summary_df],
        ignore_index=True,
    )
    run_order = [run['run_id'] for run in EXPERIMENT_RUNS]
    corrected_summary_df['_run_order'] = pd.Categorical(
        corrected_summary_df['run_id'],
        categories=run_order,
        ordered=True,
    )
    corrected_summary_df = (
        corrected_summary_df.sort_values('_run_order')
        .drop(columns='_run_order')
        .reset_index(drop=True)
    )
    CORRECTED_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    corrected_summary_df.to_csv(
        CORRECTED_SUMMARY_PATH,
        index=False,
        encoding='utf-8-sig',
    )
    LOGGER.info('修正版完整十组汇总已保存：%s', CORRECTED_SUMMARY_PATH)
    display(corrected_summary_df)


2026-08-27 00:40:59,603 | INFO | [Phase 2 1/6] 开始 fixed_random_k1


2026-08-27 00:41:01,909 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:04,709 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:07,174 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:07,868 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:08,517 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:10,760 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:13,262 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:13,909 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:41:14,571 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

2026-08-27 00:42:50,718 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:42:53,176 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:42:55,512 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:42:59,184 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:43:01,688 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:43:02,319 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:43:02,936 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:43:03,565 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:43:04,169 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

2026-08-27 00:45:09,511 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:10,174 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:10,819 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:13,789 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:16,074 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:16,828 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:17,445 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:20,574 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:45:23,314 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

2026-08-27 00:47:12,772 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:14,998 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:17,795 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:18,479 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:19,102 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:19,792 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:20,421 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:21,064 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 00:47:21,707 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 

...

2026-08-27 02:07:28,582 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 02:07:28,725 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-08-27 02:07:28,726 | WARNING | LLM 调用失败：attempt=1/3 error=Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4214>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:07:29,777 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-08-27 02:07:29,778 | WARNING | LLM 调用失败：attempt=2/3 error=Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4214>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:07:31,832 | INFO | HTTP Request: POST http://127

2026-08-27 02:07:55,994 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-08-27 02:07:55,995 | WARNING | LLM 调用失败：attempt=3/3 error=Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4147>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:07:55,996 | WARNING | 生成失败：sample_id=597 attempt=1/2 error=LLM 调用失败，已重试 3 次：Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4147>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:07:56,134 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-08-27 02:07:56,135 | WARNING | LLM 调用失败：attempt=1/3 error=Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is

2026-08-27 02:08:59,239 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-08-27 02:08:59,240 | WARNING | LLM 调用失败：attempt=1/3 error=Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4178>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:09:00,306 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-08-27 02:09:00,307 | WARNING | LLM 调用失败：attempt=2/3 error=Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4178>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:09:02,358 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-08-27 02:09:02,359 | WARNING | LLM 调用失败：attempt=

2026-08-27 02:09:48,189 | WARNING | 生成失败：sample_id=1862 attempt=2/2 error=LLM 调用失败，已重试 3 次：Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4105>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:09:48,190 | ERROR | 生成兜底：sample_id=1862 error=LLM 调用失败，已重试 3 次：Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4105>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}
2026-08-27 02:09:52,298 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 02:09:54,268 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 02:09:56,283 | INFO | HTTP Request: POST http://127.0.0.1:1234/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-27 02:09:59,667 | INFO | HTTP 

KeyboardInterrupt: 

## `fixed_rag_k5` 单独补跑

前一次运行期间，LM Studio 对部分约 4.1k–4.3k prompt tokens 的请求返回 `n_ctx=4096`，造成密集 HTTP 400 和 False 兜底，随后人工终止。该次未形成完整报告，也没有写入补跑汇总。本节只重跑 `fixed_rag_k5`，强制使用已通过长 prompt smoke 的 `context_length=8192, parallel=4`，并在加载响应和 LM Studio inventory 两处核对实例配置。


In [ ]:
# ===== fixed_rag_k5 专用补跑配置 =====
FIXED_RAG_K5_CONTEXT_LENGTH = 8192
FIXED_RAG_K5_PARALLEL = 4
FIXED_RAG_K5_SUMMARY_PATH = (
    PROJECT_ROOT / REPORT_DIR / 'phase2_qwen27_fixed_rag_k5_rerun_summary.csv'
)
RUN_FIXED_RAG_K5_RERUN = False  # 确认配置后手动改为 True。

FIXED_RAG_K5_RUN = runs_by_id['fixed_rag_k5'].copy()
display(pd.DataFrame([{
    **FIXED_RAG_K5_RUN,
    'context_length': FIXED_RAG_K5_CONTEXT_LENGTH,
    'parallel': FIXED_RAG_K5_PARALLEL,
}]))
LOGGER.info('单组补跑汇总将保存到：%s', FIXED_RAG_K5_SUMMARY_PATH)


In [ ]:
# ===== 只执行 fixed_rag_k5，并更新六组补跑与完整十组汇总 =====
if not RUN_FIXED_RAG_K5_RERUN:
    LOGGER.warning(
        'fixed_rag_k5 补跑未启动。将 RUN_FIXED_RAG_K5_RERUN 改为 True。'
    )
else:
    FIXED_RAG_K5_RESULTS = run_phase2(
        [FIXED_RAG_K5_RUN],
        summary_path=FIXED_RAG_K5_SUMMARY_PATH,
        context_length=FIXED_RAG_K5_CONTEXT_LENGTH,
        parallel=FIXED_RAG_K5_PARALLEL,
    )
    fixed_rag_k5_df = pd.DataFrame(FIXED_RAG_K5_RESULTS)
    if (
        len(fixed_rag_k5_df) != 1
        or fixed_rag_k5_df.iloc[0]['run_id'] != 'fixed_rag_k5'
        or fixed_rag_k5_df.iloc[0]['status'] != 'completed'
    ):
        raise RuntimeError('fixed_rag_k5 未完整成功，不更新正式汇总。')
    display(fixed_rag_k5_df)

    previous_rerun_df = pd.read_csv(RERUN_SUMMARY_PATH)
    previous_rerun_df = previous_rerun_df[
        previous_rerun_df['run_id'] != 'fixed_rag_k5'
    ]
    phase2_rerun_summary_df = pd.concat(
        [previous_rerun_df, fixed_rag_k5_df],
        ignore_index=True,
    )
    if set(phase2_rerun_summary_df['run_id']) != set(RERUN_RUN_IDS):
        raise RuntimeError('合并后的补跑汇总不是预期的六组配置。')
    phase2_rerun_summary_df['_run_order'] = pd.Categorical(
        phase2_rerun_summary_df['run_id'],
        categories=list(RERUN_RUN_IDS),
        ordered=True,
    )
    phase2_rerun_summary_df = (
        phase2_rerun_summary_df.sort_values('_run_order')
        .drop(columns='_run_order')
        .reset_index(drop=True)
    )
    phase2_rerun_summary_df.to_csv(
        RERUN_SUMMARY_PATH,
        index=False,
        encoding='utf-8-sig',
    )

    original_summary_df = pd.read_csv(SUMMARY_PATH)
    valid_original_df = original_summary_df[
        original_summary_df['run_id'].isin(VALID_ORIGINAL_RUN_IDS)
    ].copy()
    if set(valid_original_df['run_id']) != set(VALID_ORIGINAL_RUN_IDS):
        raise RuntimeError('原始汇总中缺少 fixed 或 RAG-only 有效结果。')
    corrected_summary_df = pd.concat(
        [valid_original_df, phase2_rerun_summary_df],
        ignore_index=True,
    )
    run_order = [run['run_id'] for run in EXPERIMENT_RUNS]
    corrected_summary_df['_run_order'] = pd.Categorical(
        corrected_summary_df['run_id'],
        categories=run_order,
        ordered=True,
    )
    corrected_summary_df = (
        corrected_summary_df.sort_values('_run_order')
        .drop(columns='_run_order')
        .reset_index(drop=True)
    )
    corrected_summary_df.to_csv(
        CORRECTED_SUMMARY_PATH,
        index=False,
        encoding='utf-8-sig',
    )
    LOGGER.info('六组补跑汇总已更新：%s', RERUN_SUMMARY_PATH)
    LOGGER.info('修正版完整十组汇总已更新：%s', CORRECTED_SUMMARY_PATH)
    display(corrected_summary_df)
